In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import linregress
import ipywidgets as widgets
from IPython.display import display, clear_output


def normalize_columns(df):
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(r"[\s\-]+", "_", regex=True)
    )
    return df


def run_pipeline(filepath, max_toggle_rate, min_toggle_rate, dead_time_ns, bin_shift,
                 bg_start_m, bg_end_m, energy_mj, blend_r1, blend_r2):
    config = {
        "bin_spacing_m":   3.75,
        "sig_start":       3840,
        "sig_end":         18836.25,
        "dead_time_ns":    dead_time_ns,
        "min_toggle_rate": min_toggle_rate,
        "max_toggle_rate": max_toggle_rate,
        "bin_shift":       bin_shift,
        "bg_start_m":      bg_start_m,
        "bg_end_m":        bg_end_m,
        "energy_mj":       energy_mj,
        "blend_r1":        blend_r1,
        "blend_r2":        blend_r2,
    }

    # ── load ──────────────────────────────────────────────────────────────────
    df = pd.read_csv(filepath)
    df = normalize_columns(df)

    # ── bin index & range ─────────────────────────────────────────────────────
    df.insert(0, "bin_index", np.arange(len(df)))
    df.insert(1, "range_m",   df["bin_index"] * config["bin_spacing_m"])

    # ── bin shift ─────────────────────────────────────────────────────────────
    df["photon_counting_shifted"] = df["photon_counting"].shift(config["bin_shift"])

    # ── dead-time correction ──────────────────────────────────────────────────
    dead_time_s  = config["dead_time_ns"] * 1e-9
    rate_meas_hz = df["photon_counting_shifted"] * 1e6
    ratio        = rate_meas_hz * dead_time_s
    df["photon_deadtime_corr"] = np.where(
        ratio < 1,
        rate_meas_hz / (1 - ratio) / 1e6,
        np.nan,
    )

    # ── signal region ─────────────────────────────────────────────────────────
    df_sig = df[
        (df["range_m"] >= config["sig_start"]) &
        (df["range_m"] <= config["sig_end"])
    ].copy()

    # re-zero range
    df_sig["bin_index"] = df_sig["bin_index"] - df_sig["bin_index"].iloc[0]
    df_sig["range_m"]   = df_sig["bin_index"] * config["bin_spacing_m"]

    # ── regression in toggle window ───────────────────────────────────────────
    # Primary: rows where photon is inside the toggle window (normal case)
    mask = (
        (df_sig["photon_deadtime_corr"] >= config["min_toggle_rate"]) &
        (df_sig["photon_deadtime_corr"] <= config["max_toggle_rate"]) &
        np.isfinite(df_sig["analog"]) &
        np.isfinite(df_sig["photon_deadtime_corr"])
    )
    m_valid = df_sig.loc[mask, ["analog", "photon_deadtime_corr"]]

    # Fallback: photon is saturated everywhere (e.g. noon) — use all finite rows
    if len(m_valid) < 2:
        mask_fallback = (
            np.isfinite(df_sig["analog"]) &
            np.isfinite(df_sig["photon_deadtime_corr"])
        )
        m_valid = df_sig.loc[mask_fallback, ["analog", "photon_deadtime_corr"]]

    slope = offset = fit_quality = np.nan
    if len(m_valid) >= 2:
        slope, offset, r_val, *_ = linregress(
            m_valid["analog"].to_numpy(),
            m_valid["photon_deadtime_corr"].to_numpy(),
        )
        fit_quality = r_val ** 2
        df_sig["analog_scaled_for_glue"] = slope * df_sig["analog"] + offset
    else:
        df_sig["analog_scaled_for_glue"] = np.nan

    # ── glue weights ──────────────────────────────────────────────────────────
    r  = df_sig["range_m"].to_numpy()
    pc = df_sig["photon_counting"].to_numpy()

    blend_r1, blend_r2 = config["blend_r1"], config["blend_r2"]
    w = np.ones(len(r), dtype=float)
    m_blend = (r >= blend_r1) & (r <= blend_r2)
    w[m_blend] = 0.5 * (
        1.0 - np.cos(np.pi * (r[m_blend] - blend_r1) / (blend_r2 - blend_r1))
    )
    w[pc > config["max_toggle_rate"]] = 0.0   # force analog
    w[pc < config["min_toggle_rate"]] = 1.0   # force photon

    df_sig["weight_w"]   = w
    df_sig["merged_sig"] = (
        (1.0 - w) * df_sig["analog_scaled_for_glue"]
        + w        * df_sig["photon_deadtime_corr"]
    )

    # ── background subtraction (on merged_sig only, after gluing) ─────────────
    bg_region = df[
        (df["range_m"] >= config["bg_start_m"]) &
        (df["range_m"] <= config["bg_end_m"]) &
        np.isfinite(df["photon_deadtime_corr"])
    ]
    merged_bg_mean = bg_region["photon_deadtime_corr"].mean()

    df_sig["merged_sig_bg_corr"] = df_sig["merged_sig"] - merged_bg_mean

    # ── range² correction + energy normalisation ──────────────────────────────
    e_joule = config["energy_mj"] * 1e-3
    df_sig["range2_corrected_counts"] = (
        df_sig["merged_sig_bg_corr"] * df_sig["range_m"] ** 2
    )
    df_sig["nrb"]        = df_sig["range2_corrected_counts"] / e_joule
    max_val              = np.nanmax(df_sig["nrb"].to_numpy())
    df_sig["range2_norm"] = df_sig["nrb"] / max_val

    return df_sig, slope, offset, fit_quality, merged_bg_mean


def make_plots(df_sig, show_merged=True, show_photon=True, show_analog=True):
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10), sharex=True)

    # ── top: glued profile (original, before bg subtraction) ──────────────────
    if show_merged:
        ax1.plot(
            df_sig["range_m"], df_sig["merged_sig"],
            color="#1a6faf", linewidth=2, linestyle="--",
            label="merged_sig (glued profile)", zorder=3,
        )
    if show_photon:
        ax1.plot(
            df_sig["range_m"], df_sig["photon_deadtime_corr"],
            color="#e05c2d", linewidth=1.2, alpha=0.85,
            label="photon_deadtime_corr", zorder=2,
        )
    if show_analog:
        ax1.plot(
            df_sig["range_m"], df_sig["analog_scaled_for_glue"],
            color="#2ca05a", linewidth=1.2, alpha=0.85,
            label="analog (scaled for glue)", zorder=2,
        )
    ax1.set_ylabel("Signal (MHz)", fontsize=13)
    ax1.set_title("LIDAR signal — merged / photon deadtime corr / analog", fontsize=14)
    ax1.set_yscale("log")
    ax1.set_xlim(0, df_sig["range_m"].max())
    ax1.grid(True, which="both", alpha=0.25)
    ax1.legend(loc="upper right", fontsize=11)

    # ── bottom: bg-subtracted → range² corrected → energy normalised ──────────
    ax2.plot(
        df_sig["range_m"], df_sig["range2_norm"],
        color="#7a3fbf", linewidth=1.8,
        label="merged_sig_bg_corr × r²  ÷  energy  ÷  max", zorder=2,
    )
    ax2.set_xlabel("Range (m)", fontsize=13)
    ax2.set_ylabel("Normalised range² counts (a.u.)", fontsize=13)
    ax2.set_title("BG subtracted → range² → energy normalisation → ÷ max", fontsize=14)
    ax2.set_xlim(0, df_sig["range_m"].max())
    ax2.axhline(0, color="gray", linewidth=0.8, linestyle=":")
    ax2.grid(True, which="both", alpha=0.25)
    ax2.legend(loc="upper right", fontsize=11)

    plt.tight_layout()
    return fig


# ─── widgets ──────────────────────────────────────────────────────────────────
style  = {"description_width": "160px"}
layout = widgets.Layout(width="420px")

w_file       = widgets.Text(
    value="", placeholder="/path/to/your/file.csv",
    description="File path:", style=style, layout=layout,
)
w_max_toggle = widgets.FloatText(
    value=10.0, step=0.5,
    description="Max toggle rate:",
    style=style, layout=layout,
)
w_min_toggle = widgets.FloatText(
    value=0.5, step=0.05,
    description="Min toggle rate:",
    style=style, layout=layout,
)
w_dead_time  = widgets.FloatText(
    value=3.06, step=0.01,
    description="Dead time (ns):",
    style=style, layout=layout,
)
w_bin_shift  = widgets.IntText(
    value=0, step=1,
    description="Bin shift:",
    style=style, layout=layout,
)
w_blend_r1   = widgets.FloatText(
    value=100.0, step=50.0,
    description="Blend start (m):",
    style=style, layout=layout,
)
w_blend_r2   = widgets.FloatText(
    value=400.0, step=50.0,
    description="Blend end (m):",
    style=style, layout=layout,
)
w_bg_start   = widgets.FloatText(
    value=0.0, step=100.0,
    description="BG start (m):",
    style=style, layout=layout,
)
w_bg_end     = widgets.FloatText(
    value=3750.0, step=100.0,
    description="BG end (m):",
    style=style, layout=layout,
)
w_energy     = widgets.FloatText(
    value=25.0, step=1.0,
    description="Energy (mJ):",
    style=style, layout=layout,
)

btn_run     = widgets.Button(
    description="Run pipeline", button_style="primary",
    layout=widgets.Layout(width="160px", height="36px"),
)
out_metrics = widgets.Output()
out_plot    = widgets.Output()

_last_df_sig = [None]  # mutable container to hold last result

btn_layout = widgets.Layout(width="160px", height="32px")
t_merged = widgets.ToggleButton(value=True, description="merged_sig",        button_style="", layout=btn_layout)
t_photon = widgets.ToggleButton(value=True, description="photon_deadtime_corr", button_style="", layout=btn_layout)
t_analog = widgets.ToggleButton(value=True, description="analog (scaled)",    button_style="", layout=btn_layout)

def _replot(*_):
    if _last_df_sig[0] is None:
        return
    with out_plot:
        clear_output(wait=True)
        fig = make_plots(
            _last_df_sig[0],
            show_merged=t_merged.value,
            show_photon=t_photon.value,
            show_analog=t_analog.value,
        )
        plt.show()
        plt.close(fig)

t_merged.observe(_replot, names="value")
t_photon.observe(_replot, names="value")
t_analog.observe(_replot, names="value")


def on_run(_):
    with out_metrics: clear_output(wait=True)
    with out_plot:    clear_output(wait=True)

    fp = w_file.value.strip()
    if not fp:
        with out_metrics: print("⚠  Please enter a file path.")
        return

    try:
        df_sig, slope, offset, fit_quality, merged_bg = run_pipeline(
            filepath         = fp,
            max_toggle_rate  = w_max_toggle.value,
            min_toggle_rate  = w_min_toggle.value,
            dead_time_ns     = w_dead_time.value,
            bin_shift        = w_bin_shift.value,
            bg_start_m       = w_bg_start.value,
            bg_end_m         = w_bg_end.value,
            energy_mj        = w_energy.value,
            blend_r1         = w_blend_r1.value,
            blend_r2         = w_blend_r2.value,
        )
    except Exception as e:
        with out_metrics: print(f"❌  Error: {e}")
        return

    with out_metrics:
        print(f"{'Slope':<20}: {slope:.6f}")
        print(f"{'Offset':<20}: {offset:.6f}")
        print(f"{'Fit quality R²':<20}: {fit_quality:.6f}")
        print(f"{'Merged BG mean':<20}: {merged_bg:.6f}")

    _last_df_sig[0] = df_sig
    with out_plot:
        fig = make_plots(
            df_sig,
            show_merged=t_merged.value,
            show_photon=t_photon.value,
            show_analog=t_analog.value,
        )
        plt.show()
        plt.close(fig)


btn_run.on_click(on_run)

display(widgets.VBox([
    widgets.HTML("<h3 style='margin:0 0 8px'>LIDAR pipeline</h3>"),
    w_file,
    widgets.HTML("<i style='font-size:12px;color:gray'>— gluing parameters —</i>"),
    w_max_toggle,
    w_min_toggle,
    w_dead_time,
    w_bin_shift,
    widgets.HTML("<i style='font-size:12px;color:gray'>— blend region —</i>"),
    w_blend_r1,
    w_blend_r2,
    widgets.HTML("<i style='font-size:12px;color:gray'>— background region —</i>"),
    w_bg_start,
    w_bg_end,
    widgets.HTML("<i style='font-size:12px;color:gray'>— normalisation —</i>"),
    w_energy,
    btn_run,
    widgets.HTML("<b>Results</b>"),
    out_metrics,
    widgets.HTML("<i style='font-size:12px;color:gray'>— toggle lines (top plot) —</i>"),
    widgets.HBox([t_merged, t_photon, t_analog]),
    out_plot,
]))